# Install Dependencies

In [ ]:
!pip install -q ktrain
!pip install -q tensorflow==2.12
!pip install -q keras==2.12
!pip install -q transformers==4.37.2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID";
os.environ["CUDA_VISIBLE_DEVICES"]="0";

# Building an Arabic Sentiment Analyzer With BERT

In [ ]:
import pandas as pd
df = pd.read_excel('All Climate Change Data - All Related.xlsx')
df = df[['text', 'Final Label']]
df.head()

,text,Final Label
0,هذه ال٢.٥٪ لا تنطلق إلى الفضاء الكوني فتحتبس و...,Negative
1,#عاجل | إدارة الكوارث والطوارئ التركية: إزالة ...,Neutral
2,RT @USUN: عُقد في مالطا هذا الأسبوع أول اجتماع...,Neutral
3,رغم ارتفاع درجات الحرارة وأشعة الشمس اللاهبة و...,Positive
4,قبل أيام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


Let's split out a training and validation set.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, temp = train_test_split(df,
                               test_size=0.3,
                               stratify=df['Final Label'],
                               random_state=42)

df_val, df_test = train_test_split(temp,
                             test_size=0.5,
                             stratify=temp['Final Label'],
                             random_state=42)

In [ ]:
import ktrain
from ktrain import text

In [ ]:
import ktrain
from ktrain import text
MODEL_NAME = 'aubmindlab/bert-base-arabertv02-twitter'
t = text.Transformer(MODEL_NAME, maxlen=128)
trn = t.preprocess_train(df_train['text'].values, df_train['Final Label'].values)
val = t.preprocess_test(df_val['text'].values, df_val['Final Label'].values)
model = t.get_classifier()
learner = ktrain.get_learner(model, train_data=trn, val_data=val, batch_size=32)
learner.fit_onecycle(5e-5, 1)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

preprocessing train...
language: ar
train sequence lengths:
	mean : 25
	95percentile : 47
	99percentile : 52


tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Is Multi-Label? False
preprocessing test...
language: ar
test sequence lengths:
	mean : 25
	95percentile : 47
	99percentile : 52




begin training using onecycle policy with max lr of 5e-05...
525/525 [==============================] - 2508s 5s/step - loss: 0.6664 - accuracy: 0.6975 - val_loss: 0.5829 - val_accuracy: 0.7318


### Making Predictions on ClimaSentAR

In [ ]:
p = ktrain.get_predictor(learner.model, t)

In [ ]:
# save model for later use
p.save('arabic_predictor')

In [ ]:
pred_climate = []
for text in df_test['text']:
  pred_climate.append(p.predict(text))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(df_test['Final Label'],pred_climate, digits=4))

              precision    recall  f1-score   support

    Negative     0.7348    0.8190    0.7746      1221
     Neutral     0.7479    0.6328    0.6856      1547
    Positive     0.7284    0.8148    0.7691       826

    accuracy                         0.7379      3594
   macro avg     0.7370    0.7555    0.7431      3594
weighted avg     0.7389    0.7379    0.7350      3594



### Save our Predictor for Later Deployment

In [ ]:
# reload from disk
p = ktrain.load_predictor('arabic_predictor')

# ASTD

In [ ]:
import pandas as pd

# Read the .txt file (comma, tab, or custom-delimited)
df = pd.read_csv('Tweets.txt', delimiter='\t', header=None, names=['text', 'Label'])  # Change delimiter if needed
df.head()

,text,Label
0,بعد استقالة رئيس #المحكمة_الدستورية ننتظر استق...,OBJ
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,POS
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,NEG
3,#الحرية_والعدالة | شاهد الآن: #ليلة_الاتحادية ...,OBJ
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,NEUTRAL


In [ ]:
df = df[df['Label']!='OBJ']
df['Label'] = df['Label'].map({
    'NEG': 'Negative',
    'NEUTRAL': 'Neutral',
    'POS': 'Positive'
})
df.head()

,text,Label
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,Positive
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,Negative
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,Neutral
5,#انتخبوا_العرص #انتخبوا_البرص #مرسى_رئيسى #اين...,Neutral
6,امير عيد هو اللي فعلا يتقال عليه ستريكر صريح #...,Positive


In [ ]:
df.dropna(inplace = True)
df = df.drop_duplicates(subset = 'text')

In [ ]:
df.shape

(3224, 2)

In [ ]:
pred_astd = []
for text in df['text']:
  pred_astd.append(p.predict(text))

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(df['Label'],pred_astd, digits=4))

              precision    recall  f1-score   support

    Negative     0.7828    0.6322    0.6995      1642
     Neutral     0.3348    0.6596    0.4442       805
    Positive     0.8814    0.3539    0.5051       777

    accuracy                         0.5720      3224
   macro avg     0.6663    0.5486    0.5496      3224
weighted avg     0.6947    0.5720    0.5889      3224



# SemEval

In [ ]:
semeval = pd.read_csv('SemEval2017-task4-train.subtask-A.arabic.txt', delimiter='\t', header=None, names=['id', 'sentiment', 'text'])
semeval = semeval[['text', 'sentiment']]

semeval['sentiment'] = semeval['sentiment'].map({
    'negative': 'Negative',
    'neutral': 'Neutral',
    'positive': 'Positive'
})
semeval.head()

,text,sentiment
0,إجبار أبل على التعاون على فك شفرة اجهزتها http...,Positive
1,RT @20fourMedia: #غوغل تتحدى أبل وأمازون بأجهز...,Positive
2,جوجل تنافس أبل وسامسونج بهاتف جديد https://t.c...,Positive
3,رئيس شركة أبل: الواقع المعزز سيصبح أهم من الطع...,Positive
4,ساعة أبل في الأسواق مرة أخرى https://t.co/dY2x...,Neutral


In [ ]:
semeval.shape

(3353, 2)

In [ ]:
semeval.dropna(inplace = True)
semeval = semeval.drop_duplicates(subset = 'text')

In [ ]:
pred_semeval = []
for text in semeval['text']:
  pred_semeval.append(p.predict(text))

In [ ]:
print(classification_report(semeval['sentiment'],pred_semeval, digits=4))

              precision    recall  f1-score   support

    Negative     0.8209    0.5709    0.6735      1100
     Neutral     0.5530    0.9059    0.6868      1456
    Positive     0.7914    0.1501    0.2523       733

    accuracy                         0.6254      3289
   macro avg     0.7218    0.5423    0.5375      3289
weighted avg     0.6957    0.6254    0.5855      3289



# Pred on Asad

In [ ]:
import pandas as pd
asad = pd.read_csv('train_all.csv')

asad.dropna(inplace = True)
asad = asad.drop_duplicates(subset = 'Text')

In [ ]:
pred_asad = []
for text in asad['Text']:
  pred_asad.append(p.predict(text))

In [ ]:
asad['sentiment'] = asad['sentiment'].str.title()
print(classification_report(asad['sentiment'],pred_asad, digits=4))

              precision    recall  f1-score   support

    Negative     0.4991    0.6775    0.5748      8748
     Neutral     0.8132    0.7911    0.8020     36938
    Positive     0.5808    0.4360    0.4981      8510

    accuracy                         0.7170     54196
   macro avg     0.6310    0.6349    0.6249     54196
weighted avg     0.7260    0.7170    0.7176     54196

